# 5. BM25 + Phi-3 summary and evidence

Prompting improves precision; many invalid outputs fall back to original passages. Historical acceptance decision is preserved.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(5)
model['config_snapshot']

{'name': 'phi3_summary_with_evidence',
 'provider': 'ollama',
 'task': 'summary_with_evidence',
 'model': 'phi3:mini',
 'model_revision': '4f222292793889a9a40a020799cfd28d53f3e01af25d48e06c5e708610fc47e9',
 'runtime': 'ollama-0.40.13',
 'prompt_file': 'prompts/summarise_with_evidence_v1.txt',
 'temperature': 0,
 'seed': 42,
 'max_output_tokens': 768,
 'context_length': 12288,
 'candidate_k': 5,
 'retrieval_config': 'llm_configs/local_phi3.json',
 'max_evidence_quotes': 3,
 'max_summary_words': 45,
 'invalid_output_policy': 'return_original_candidates'}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.282986,0.465996,0.564286
1,3,140,0.226862,0.620845,0.714286
2,5,140,0.200822,0.689211,0.778571
3,10,140,0.200822,0.689211,0.778571


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 7}
False answerable abstention: 0.02142857142857143
Cached latency: {'median': 12.991600029636174, 'p95': 16.41446999274194}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
2 passages retrieved.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


Full code walkthrough: ../experiments/14_phi3_full_summary.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.